# Tahap 1 — Normalisasi Bahasa Indonesia

Notebook ini menjalankan normalisasi pada field `question` dan `answer` dari data konsultasi Alodokter. Hasilnya mencakup teks asli, teks ternormalisasi, segmentasi kalimat, audit perubahan, warning ambiguitas, dan boilerplate jawaban.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from collections import Counter
from pathlib import Path

# Notebook dapat dijalankan dari root repositori atau dari folder Riset.
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "rag").exists() else cwd.parent
if not (REPO_ROOT / "rag").exists():
    raise FileNotFoundError("Root repositori tidak ditemukan. Jalankan notebook dari root atau folder Riset.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rag.id_preprocess import IndonesianClinicalNormalizer

INPUT_FILE = REPO_ROOT / "Riset/crawler_alodokter/hasil_crawl_alodokter_qa_pairs.json"
OUTPUT_FILE = REPO_ROOT / "Riset/crawler_alodokter/hasil_normalisasi_tahap_1.jsonl"
RESOURCE_DIR = REPO_ROOT / "data/input"

print("Repository :", REPO_ROOT)
print("Input      :", INPUT_FILE)
print("Output     :", OUTPUT_FILE)
print("Resources  :", RESOURCE_DIR)

## 1. Validasi input dan resource

In [ ]:
required_files = [
    INPUT_FILE,
    RESOURCE_DIR / "id_abbreviations_layered.json",
    RESOURCE_DIR / "id_abbreviations.json",
    RESOURCE_DIR / "id_typos.json",
    RESOURCE_DIR / "id_units.json",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"File yang diperlukan tidak ditemukan: {missing}")

records = json.loads(INPUT_FILE.read_text(encoding="utf-8"))
if not isinstance(records, list):
    raise ValueError("Input harus berupa JSON array.")

normalizer = IndonesianClinicalNormalizer.from_resource_dir(RESOURCE_DIR)
print(f"Jumlah record     : {len(records)}")
print(f"Versi normalizer : 1.0.0")
print(f"Versi resource   : {normalizer.resource_version}")

## 2. Demonstrasi normalisasi

Profil `question` mengaktifkan normalisasi bahasa informal. Profil `answer` lebih konservatif dan memisahkan sapaan/penutup ke field `boilerplate`.

In [ ]:
contoh = "Sy skrg ISK, tp blm minum obat 500mg. Dok, kira2 berbahaya ngga?"
demo = normalizer.normalize(contoh, audit=True, profile="question")

print("ASLI       :", demo.original_text)
print("NORMALISASI:", demo.normalized_text)
print("KALIMAT    :", demo.sentences)
print("WARNING    :", demo.warnings)
print("PERUBAHAN:")
for change in demo.changes:
    print(f"- {change.kind}: {change.source!r} -> {change.target!r} ({change.count}x)")

## 3. Fungsi serialisasi hasil

In [ ]:
def result_payload(result):
    return {
        "raw_text": result.original_text,
        "normalized_text": result.normalized_text,
        "content_text": result.content_text,
        "sentences": result.sentences,
        "normalization_changes": [change.__dict__ for change in result.changes],
        "warnings": result.warnings,
        "boilerplate": result.boilerplate,
        "replacements": result.replacements,
        "profile": result.profile,
        "normalizer_version": result.normalizer_version,
        "resource_version": result.resource_version,
    }


def normalize_record(row, index):
    question = row.get("question") or {}
    answer = row.get("answer") or {}
    question_text = question.get("raw_text") or question.get("clean_text") or ""
    answer_text = answer.get("raw_text") or answer.get("clean_text") or ""

    question_result = normalizer.normalize(question_text, audit=True, profile="question")
    answer_result = normalizer.normalize(answer_text, audit=True, profile="answer")
    record_id = hashlib.sha256(str(row.get("url") or index).encode("utf-8")).hexdigest()[:16]

    return {
        "record_id": record_id,
        "source": row.get("source"),
        "url": row.get("url"),
        "title": row.get("title"),
        "question": result_payload(question_result),
        "answer": result_payload(answer_result),
    }

## 4. Jalankan Tahap 1 untuk seluruh data

In [ ]:
normalized_records = [normalize_record(row, index) for index, row in enumerate(records, start=1)]

if len({row["record_id"] for row in normalized_records}) != len(normalized_records):
    raise ValueError("Ditemukan record_id duplikat.")

print(f"Berhasil menormalisasi {len(normalized_records)} record.")

## 5. Ringkasan audit

In [ ]:
summary = {
    "records": len(normalized_records),
    "question_change_groups": sum(len(row["question"]["normalization_changes"]) for row in normalized_records),
    "answer_change_groups": sum(len(row["answer"]["normalization_changes"]) for row in normalized_records),
    "warnings": sum(len(row["question"]["warnings"]) + len(row["answer"]["warnings"]) for row in normalized_records),
    "question_sentences": sum(len(row["question"]["sentences"]) for row in normalized_records),
    "answer_sentences": sum(len(row["answer"]["sentences"]) for row in normalized_records),
}
summary

In [ ]:
change_counter = Counter()
for row in normalized_records:
    for field in ("question", "answer"):
        for change in row[field]["normalization_changes"]:
            change_counter[(change["kind"], change["source"], change["target"])] += change["count"]

print("20 perubahan paling sering:")
for (kind, source, target), count in change_counter.most_common(20):
    print(f"{count:>3}x | {kind:<13} | {source!r} -> {target!r}")

## 6. Inspeksi warning dan contoh hasil

In [ ]:
warning_rows = []
for row in normalized_records:
    for field in ("question", "answer"):
        if row[field]["warnings"]:
            warning_rows.append({
                "record_id": row["record_id"],
                "field": field,
                "warnings": row[field]["warnings"],
                "normalized_text": row[field]["normalized_text"],
            })

print(f"Record/field dengan warning: {len(warning_rows)}")
warning_rows[:3]

In [ ]:
sample_index = 1
sample = normalized_records[sample_index]
print("JUDUL:", sample["title"])
print("\nPERTANYAAN ASLI:\n", sample["question"]["raw_text"])
print("\nPERTANYAAN NORMALISASI:\n", sample["question"]["normalized_text"])
print("\nISI JAWABAN TANPA BOILERPLATE:\n", sample["answer"]["content_text"][:1000])

## 7. Simpan artefak JSONL

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_FILE.open("w", encoding="utf-8", newline="\n") as handle:
    for row in normalized_records:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Tersimpan: {OUTPUT_FILE}")
print(f"Ukuran   : {OUTPUT_FILE.stat().st_size:,} byte")

## 8. Pemeriksaan akhir

In [ ]:
saved_rows = [json.loads(line) for line in OUTPUT_FILE.read_text(encoding="utf-8").splitlines() if line]
assert len(saved_rows) == len(records)
assert len({row["record_id"] for row in saved_rows}) == len(saved_rows)
assert all(row["question"]["raw_text"] is not None for row in saved_rows)
assert all(row["answer"]["normalized_text"] is not None for row in saved_rows)

print("Validasi akhir berhasil.")
print(json.dumps(summary, indent=2, ensure_ascii=False))